# Progress Prediction Training Example

This notebook allows you to:
1. Connect to the R2 Robot backend, which includes a progress prediction training endpoint.
2. Start training jobs for progress prediction models.
3. Monitor training progress with live metrics (loss, accuracy, F1).
4. Cancel training if needed.

Progress prediction models predict task completion progress (0-1) from camera images,
used for behavior termination detection.

Uses the `ProgressPredictionTrainerClient` from the main SDK client module (`r2_labs.sdk.client`).

Remember to start the R2 backend first before running this notebook.

In [ ]:
import time
from IPython.display import clear_output, display

from r2_labs.rpc import client as rpc_client
from r2_labs.sdk import client as sdk_client
from r2_labs.sdk import rpc_api
from r2_labs.sdk.progress_widget import TrainingProgressWidget, monitor_training

%load_ext autoreload
%autoreload 2

## 1. Configure Server Connection

Set the hostname/IP of the machine running the R2 backend.

In [ ]:
# Configure the server address
TRAINING_SERVER_HOST = "cortex-lon1.local"  # Change to your server IP/hostname
TRAINING_SERVER_PORT = rpc_api.DEFAULT_MODEL_TRAINER_PORT  # 7534

server_address = f"tcp://{TRAINING_SERVER_HOST}:{TRAINING_SERVER_PORT}"
print(f"Training server: {server_address}")

## 2. Connect to the Server

Create a `ProgressPredictionTrainerClient` using the SDK client module.

In [ ]:
# Create the base RPC client
base_client = rpc_client.BaseClient(
    server_address,
    timeout=3_000,  # 30 second timeout
)

# Create the progress prediction trainer client from the SDK
trainer = sdk_client.ProgressPredictionTrainerClient(base_client)

# Test connection by getting status
try:
    status = trainer.get_training_status()
    print("Connected to Progress Training server!")
except Exception as e:
    print(f"Failed to connect: {e}")

## 3. Check Current Training Status

Use this to check if training is already running.

In [ ]:
def print_status(status):
    """Pretty print training status."""
    print("=" * 50)
    print(f"Phase: {status.phase}")
    print(f"Is Finished: {status.is_finished}")
    print(f"Steps: {status.steps_completed} / {status.max_steps}")
    if status.loss is not None:
        print(f"Loss: {status.loss:.6f}")
    if status.accuracy is not None:
        print(f"Accuracy: {status.accuracy:.4f}")
    if status.f1 is not None:
        print(f"F1 Score: {status.f1:.4f}")
    if status.fps is not None:
        print(f"Speed: {status.fps:.2f} steps/sec")
    if status.seconds_per_step is not None:
        print(f"Time per step: {status.seconds_per_step:.3f} sec")
    if status.error:
        print(f"Error: {status.error}")
    print("=" * 50)

# Check current status
status = trainer.get_training_status()
print_status(status)

## 4. Start Training

### Option A: Train from full episodes

Use `entry_filters` to match full episodes from the data warehouse.

In [ ]:
# Training configuration with entry filters
MODEL_NAME = "progress_model"
TRAINING_STEPS = 10_000
# Glob patterns to match data warehouse entries
ENTRY_FILTERS = ["rectify_open_latch_gravcomp*",
                 "rectify_open_latch_scene_camera*",
                 "rectify_open_latch_fixed*"]

# Start training with entry_filters
response = trainer.train_model(
    model_name=MODEL_NAME,
    training_steps=TRAINING_STEPS,
    entry_filters=ENTRY_FILTERS,
    batch_size=32,
    task_type="classification",  # or "regression" for continuous 0-1 progress
    cameras=["right_camera", "wrist_camera"],
    force_rebuild=False,  # Set to True to force rebuild the dataset
)

if response.error:
    print(f"ERROR: {response.error}")
else:
    print("Training started successfully!")
    # Start monitoring widget - updates automatically until training completes
    widget = TrainingProgressWidget(trainer, poll_interval=2.0)
    widget.start()

### Option B: Train from human demonstrations (DAgger data)

Use `human_entry_filters` to extract only human segments from episodes.

In [ ]:
# Training configuration with human entry filters
MODEL_NAME = "progress_model_human"
TRAINING_STEPS = 10000
HUMAN_ENTRY_FILTERS = ["dagger_*"]  # Glob patterns for human demonstration entries

# Start training with human_entry_filters
response = trainer.train_model(
    model_name=MODEL_NAME,
    training_steps=TRAINING_STEPS,
    human_entry_filters=HUMAN_ENTRY_FILTERS,
    batch_size=32,
    task_type="classification",
    cameras=["wrist_camera"],
    force_rebuild=False,
)

if response.error:
    print(f"ERROR: {response.error}")
else:
    print("Training started successfully!")
    widget = TrainingProgressWidget(trainer, poll_interval=2.0)
    widget.start()

### Option C: Mix of full episodes and human demonstrations

Combine both `entry_filters` and `human_entry_filters` for mixed training data.

In [ ]:
# Training configuration with mixed filters
MODEL_NAME = "progress_model_mixed"
TRAINING_STEPS = 10000
ENTRY_FILTERS = ["agent_demos*"]  # Full episodes
HUMAN_ENTRY_FILTERS = ["dagger_pick*", "dagger_place*"]  # Human segments

# Start training with both filters
response = trainer.train_model(
    model_name=MODEL_NAME,
    training_steps=TRAINING_STEPS,
    entry_filters=ENTRY_FILTERS,
    human_entry_filters=HUMAN_ENTRY_FILTERS,
    batch_size=32,
    task_type="classification",
    cameras=["wrist_camera"],
    force_rebuild=False,
)

if response.error:
    print(f"ERROR: {response.error}")
else:
    print("Training started successfully!")
    widget = TrainingProgressWidget(trainer, poll_interval=2.0)
    widget.start()

## 5. Monitor Training Progress

Use the `TrainingProgressWidget` for live progress display with metrics.

**Option A:** One-liner that blocks until training completes:
```python
final_status = monitor_training(trainer)
```

**Option B:** Manual control for more flexibility (shown below):

In [ ]:
# Create and start the progress widget
widget = TrainingProgressWidget(trainer, poll_interval=2.0)
widget.start()

# The widget updates automatically in the background.
# Call widget.stop() to stop polling, or just let it run until training finishes.

## 6. Export Model

Export the current model from the latest checkpoint to the model warehouse.

In [ ]:
# Export the model
response = trainer.export_model()

if response.success:
    print(f"Model exported successfully!")
    print(f"Model version: {response.model_id}")
else:
    print(f"Failed to export: {response.error}")

## 7. Cancel Training

Use this to stop training early if needed.

In [ ]:
# Cancel training (optionally export model before cancelling)
response = trainer.cancel_training(export_model=False)

if response.success:
    print("Training cancelled successfully!")
else:
    print(f"Failed to cancel: {response.error}")

## 8. One-time Status Check

In [ ]:
# Quick status check
status = trainer.get_training_status()
print_status(status)